# Day 211 — Inference Wrapper (TeleServe India)
### Month 13 · Week 3 · Day 5 — Agentic AI & Advanced GenAI Portfolio

**Prerequisite:** Day210's saved LoRA adapter (`teleserve_lora_adapter_209` on Google Drive) and its confidence-separation finding (avg conf 0.8766 correct vs 0.6401 incorrect → 0.60 threshold).

**Today's job is narrow on purpose:** you already trained the model (Day209) and evaluated it honestly (Day210). Day211 does **no new modeling**. It takes what exists and wraps it into one clean, reusable, production-shaped function:

```
ticket_text (str)  -->  {priority, confidence, routing_decision, ...}
```

This function is the exact component Days212–216's capstone chatbot will import and call — so today's real deliverable is a well-behaved Python **module**, not just a notebook cell.


## Concept Notes — What is an "inference wrapper" and why it's its own lesson day

**The problem it solves:** A trained model on its own is not a product. `model(**tokenizer(text))` returns raw logits — a tensor. Nothing in a real system (a chatbot, an API endpoint, a CrewAI tool) wants raw logits. They want a stable, documented contract: *given text, give me a decision, and tell me how sure you are.*

**Why this is a distinct skill from training:**
| Training (Day208-210) | Inference wrapping (Day211) |
|---|---|
| Optimized for correctness on held-out data | Optimized for **predictable behavior on ANY input**, including garbage |
| Can crash, can be slow, can print debug logs | Must never crash on bad input; must fail *informatively* |
| Runs once (or a few times) under your control | Runs every time a real ticket arrives, unattended |
| Output = accuracy metric | Output = a decision another system (or the capstone agent) acts on |

**Core design principles applied today:**
1. **Single responsibility** — the function does exactly one thing: text in, decision out. No side effects (no printing, no file writes) unless explicitly asked.
2. **Fail loud on real bugs, fail soft on bad input.** A malformed input (empty string, wrong type, absurdly long text) should return a structured `{"error": ...}` dict, not raise an uncaught exception that would crash the capstone agent's tool call.
3. **Confidence is not decoration — it's a routing signal.** Day210 proved fine-tuned accuracy drops from 100% (val) to 60% (unseen/hard) — so the wrapper cannot just return the top class. It must also decide, using Day210's real confidence-separation numbers, whether a human should see this ticket before it's acted on.
4. **The wrapper is a module, not a notebook cell.** Days212-216 will `from day211_inference_wrapper import predict_ticket_priority` — if today's code only exists as inline cells, it can't be imported. Packaging it correctly IS part of the task, not an afterthought.

**Real-world parallel:** this is exactly the shape of a "model serving" layer in production ML — the part that sits between a trained checkpoint (data science artifact) and an application (software engineering artifact). Freelance/consulting value is very often earned right here, not in the training loop.


## Section 1 — Raw Data / Artifacts (never modify this cell's output)

Two things are "raw" today:
1. **The trained artifact** — Day209's adapter, saved to Drive during Day210 (survives runtime resets).
2. **A brand-new, never-before-seen test set** (seed=211, 20 tickets) — deliberately NOT reused from Day209/210, so today's wrapper is tested on genuinely fresh input, the way it will be in the Day212-216 capstone.


In [12]:
# ── Setup: mount Drive, load base model + Day209/210 adapter ──
from google.colab import drive
drive.mount('/content/drive')

ADAPTER_PATH = "/content/drive/MyDrive/teleserve_lora_adapter_209"  # standing save location (Day210 lesson)
BASE_MODEL   = "distilbert-base-uncased"   # LoRA never modifies the tokenizer — load tokenizer from base, not adapter path

import os
assert os.path.exists(ADAPTER_PATH), (
    "Adapter not found. If this is a fresh runtime, re-run Day210's retrain-and-save-to-Drive "
    "cell first — local-disk adapters do NOT survive a Colab runtime reset (Day210 lesson)."
)
print("Adapter found at:", ADAPTER_PATH)
print(os.listdir(ADAPTER_PATH))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Adapter found at: /content/drive/MyDrive/teleserve_lora_adapter_209
['README.md', 'adapter_model.safetensors', 'adapter_config.json']


In [13]:
# ── RAW DATA (immutable): 20 fresh TeleServe India tickets, seed=211 ──
# Never seen during Day209 training or Day210 evaluation.
# Mix: 14 templated (systematic coverage of P1/P2/P3) + 6 hand-written (no template, harder).
# Ground-truth labels are provided ONLY for scoring the wrapper's real-world behavior in Task 4 —
# in a real deployment, the wrapper obviously would not receive these labels.

import random
random.seed(211)

P1_TEMPLATES = [
    "Complete network outage in {area} since {time}, no signal at all",
    "Emergency: fraudulent charge of Rs.{amt} on my account, need immediate action",
    "My connection has been down for {hrs} hours, this is affecting my work",
]
P2_TEMPLATES = [
    "I was billed Rs.{amt} extra this cycle, please explain the charge",
    "My data speed has dropped significantly since the last plan renewal",
    "Requesting a refund for the {days}-day service interruption last week",
]
P3_TEMPLATES = [
    "I'd like to upgrade my plan to the {plan} package, please share steps",
    "New SIM card needs activation, purchased {days} days ago",
    "Can you send me details on the family data-sharing plans available",
]

areas = ["Sector 14", "Model Town", "Civil Lines", "Gandhi Nagar"]
plans = ["unlimited", "premium", "family", "postpaid"]

def fill(t):
    return t.format(area=random.choice(areas), time=random.choice(["6 AM","last night","noon"]),
                     amt=random.randint(200,2500), hrs=random.randint(3,30),
                     days=random.randint(1,10), plan=random.choice(plans))

templated = []
for templates, label in [(P1_TEMPLATES,"P1"), (P2_TEMPLATES,"P2"), (P3_TEMPLATES,"P3")]:
    for _ in range(5 if label=="P1" else 5 if label=="P2" else 4):
        templated.append({"text": fill(random.choice(templates)), "true_priority": label})

hand_written = [
    {"text": "hi so like my internet just stopped working and I have a client call in 10 min pls help asap!!", "true_priority": "P1"},
    {"text": "not sure who to ask but is there a way to see all my past bills in one place", "true_priority": "P3"},
    {"text": "the guy who came to fix my line yesterday said it's fixed but it's not, still no dial tone", "true_priority": "P1"},
    {"text": "saw an extra 499 rupees on my statement, don't remember agreeing to any addon tbh", "true_priority": "P2"},
    {"text": "just moved to a new flat, want to know if i can shift my existing connection here", "true_priority": "P3"},
    {"text": "network's been spotty on and off all week, not fully down but annoying during calls", "true_priority": "P2"},
]

day211_test_set = templated + hand_written
random.shuffle(day211_test_set)

print(f"Day211 fresh test set: {len(day211_test_set)} tickets "
      f"({len(templated)} templated + {len(hand_written)} hand-written)")
for row in day211_test_set[:3]:
    print(row)

Day211 fresh test set: 20 tickets (14 templated + 6 hand-written)
{'text': 'just moved to a new flat, want to know if i can shift my existing connection here', 'true_priority': 'P3'}
{'text': 'My connection has been down for 22 hours, this is affecting my work', 'true_priority': 'P1'}
{'text': 'My data speed has dropped significantly since the last plan renewal', 'true_priority': 'P2'}


## Section 2 — Practice Guide (100 pts + 10★ bonus)

Attempt every task yourself before looking at Section 4 (Answer Key). Fill in the `# TODO` blocks.


### Task 1 (20 pts) — Load the model + adapter, reconstruct Day210's config

Load `distilbert-base-uncased` with the SAME `BitsAndBytesConfig` pattern as Day210 (including `llm_int8_skip_modules=["pre_classifier","classifier"]` — the standing fix), attach the PEFT adapter from `ADAPTER_PATH`, set `.eval()`. Also hard-code Day210's two config constants pulled from real printed output, not memory: the label map and the confidence threshold.


In [14]:
# ── Upgrade bitsandbytes to the required version ────────────────────
!pip install -U bitsandbytes>=0.46.1

In [15]:
# ── TASK 1 (20 pts): Load model + adapter, reconstruct Day210 config ──
# Goal: Load the base model with 4‑bit quantization (matching Day210) and attach
#       the saved LoRA adapter. Also set the label map and confidence threshold
#       derived from Day210's real evaluation numbers.
# Method: Use BitsAndBytesConfig with llm_int8_skip_modules, load base model,
#         apply PeftModel.from_pretrained, and set .eval().

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, BitsAndBytesConfig
from peft import PeftModel

# ---- Constants from Day210 (verified from printed output) ----
# id2label: int → str, label2id: str → int
ID2LABEL = {0: "P1", 1: "P2", 2: "P3"}
LABEL2ID = {v: k for k, v in ID2LABEL.items()}   # {"P1":0, "P2":1, "P3":2}
CONFIDENCE_THRESHOLD = 0.60   # from Day210 bonus: avg conf correct 0.8766, incorrect 0.6401

# ---- Tokenizer (always from base model, not adapter path) ----
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

# ---- BitsAndBytes config (same as Day210) ----
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    llm_int8_skip_modules=["pre_classifier", "classifier"],  # keep trained head in full precision
)

# ---- Load base model with quantization ----
base_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=3,
    id2label=ID2LABEL,          # correct: int → str
    label2id=LABEL2ID,          # correct: str → int
    quantization_config=bnb_config,
    device_map="auto",
)

# ---- Attach LoRA adapter ----
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()
torch.cuda.empty_cache()

print("✅ Model + adapter loaded successfully.")
print(f"Model device: {next(model.parameters()).device}")
print(f"Confidence threshold: {CONFIDENCE_THRESHOLD}")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Model + adapter loaded successfully.
Model device: cpu
Confidence threshold: 0.6


### Task 2 (25 pts) — Core `predict_ticket_priority()` function

Write the single function that is the heart of today's deliverable. Given raw ticket text, it must:
- Tokenize, run a forward pass, softmax the logits
- Return `priority` (top class), `confidence` (top class prob, rounded), `all_probs` (dict of all 3 class probs), `routing_decision` (`"human_review"` if `confidence < CONFIDENCE_THRESHOLD` else `"auto_route"`)
- Use `torch.no_grad()` — this is inference only, never training

Test it on 3 tickets from `day211_test_set` and print the full returned dict for each.


In [16]:
# ── TASK 2 (25 pts): Core predict_ticket_priority() function ──────
# Goal: Implement the inference function that returns a structured decision
#       (priority, confidence, all_probs, routing_decision) for a single text.
# Method: Tokenize input, run model under torch.no_grad(), apply softmax,
#         extract top class and its probability, compare confidence to threshold.

import torch.nn.functional as F

def predict_ticket_priority(text: str) -> dict:
    """
    Predict priority and confidence for a ticket text.

    Args:
        text (str): Raw customer ticket text.

    Returns:
        dict: {
            "priority": str,           # "P1", "P2", or "P3"
            "confidence": float,       # softmax probability of predicted class (rounded)
            "all_probs": dict,         # {"P1": prob, "P2": prob, "P3": prob}
            "routing_decision": str    # "auto_route" or "human_review"
        }
    """
    # Tokenize
    inputs = tokenizer(text, truncation=True, padding=True, max_length=64, return_tensors="pt")
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}

    with torch.no_grad():
        logits = model(**inputs).logits
        probs = F.softmax(logits, dim=-1).cpu().numpy()[0]   # shape (3,)

    # Convert to dictionary using ID2LABEL (int → str)
    probs_dict = {ID2LABEL[i]: float(probs[i]) for i in range(3)}
    pred_label = max(probs_dict, key=probs_dict.get)
    confidence = probs_dict[pred_label]
    routing_decision = "human_review" if confidence < CONFIDENCE_THRESHOLD else "auto_route"

    return {
        "priority": pred_label,
        "confidence": round(confidence, 4),
        "all_probs": probs_dict,
        "routing_decision": routing_decision,
    }

# ---- Quick test on 3 tickets from the fresh set ----
print("\n--- Task 2: Demo predictions ---")
for ticket in day211_test_set[:3]:
    result = predict_ticket_priority(ticket["text"])
    print(f"Text: {ticket['text'][:60]}...")
    print(f"Result: {result}\n")


--- Task 2: Demo predictions ---
Text: just moved to a new flat, want to know if i can shift my exi...
Result: {'priority': 'P3', 'confidence': 0.9854, 'all_probs': {'P1': 0.000811042613349855, 'P2': 0.01381854061037302, 'P3': 0.985370397567749}, 'routing_decision': 'auto_route'}

Text: My connection has been down for 22 hours, this is affecting ...
Result: {'priority': 'P1', 'confidence': 0.8135, 'all_probs': {'P1': 0.8135278820991516, 'P2': 0.08506041020154953, 'P3': 0.10141170024871826}, 'routing_decision': 'auto_route'}

Text: My data speed has dropped significantly since the last plan ...
Result: {'priority': 'P3', 'confidence': 0.6847, 'all_probs': {'P1': 0.0015913438983261585, 'P2': 0.3137049376964569, 'P3': 0.6847037076950073}, 'routing_decision': 'auto_route'}



### Task 3 (20 pts) — Input validation + edge cases

A real caller (the Day212-216 agent) will not always send clean text. Extend `predict_ticket_priority()` (or wrap it) to handle, WITHOUT raising an exception:
- Non-string input → `{"error": "input must be a string", ...}`
- Empty / whitespace-only string → `{"error": "input too short..."}`
- Extremely long input (>2000 chars) → truncate to 2000 chars, run prediction anyway, but set `"truncated_input": True` in the result so the caller knows

Test all 4 edge cases explicitly and print each result.


In [17]:
# ── TASK 3 (20 pts): Input validation + edge cases ──────────────────
# Goal: Extend the wrapper to handle malformed inputs safely without crashing,
#       returning structured error dicts instead.
# Method: Wrap the core predict function with type checks, emptiness, and truncation.

MAX_INPUT_LENGTH = 2000

def safe_predict_ticket_priority(text) -> dict:
    """
    Wrapper for predict_ticket_priority with input validation and graceful error handling.
    """
    # 1. Non‑string input
    if not isinstance(text, str):
        return {
            "error": "input must be a string",
            "priority": None,
            "confidence": None,
            "all_probs": None,
            "routing_decision": None,
            "truncated_input": False
        }

    # 2. Empty / whitespace‑only string
    stripped = text.strip()
    if len(stripped) == 0:
        return {
            "error": "input too short (empty or whitespace)",
            "priority": None,
            "confidence": None,
            "all_probs": None,
            "routing_decision": None,
            "truncated_input": False
        }

    # 3. Extremely long input – truncate and set flag
    truncated = False
    if len(stripped) > MAX_INPUT_LENGTH:
        stripped = stripped[:MAX_INPUT_LENGTH]
        truncated = True

    # 4. Call the core predictor
    result = predict_ticket_priority(stripped)
    result["truncated_input"] = truncated
    return result

# ---- Test all 4 edge cases ----
print("\n--- Task 3: Edge case tests ---")

# 1) Non‑string
print("1. Input = 12345")
print(safe_predict_ticket_priority(12345), "\n")

# 2) Empty string
print("2. Input = ''")
print(safe_predict_ticket_priority(""), "\n")

# 3) Very short string (OK, but we still test)
print("3. Input = 'ok'")
print(safe_predict_ticket_priority("ok"), "\n")

# 4) Long string (>2000 chars)
long_text = "x" * 3000
print(f"4. Input length = {len(long_text)} (truncated to {MAX_INPUT_LENGTH})")
result_long = safe_predict_ticket_priority(long_text)
print(f"truncated_input = {result_long['truncated_input']}")
print(f"priority = {result_long['priority']}\n")


--- Task 3: Edge case tests ---
1. Input = 12345
{'error': 'input must be a string', 'priority': None, 'confidence': None, 'all_probs': None, 'routing_decision': None, 'truncated_input': False} 

2. Input = ''
{'error': 'input too short (empty or whitespace)', 'priority': None, 'confidence': None, 'all_probs': None, 'routing_decision': None, 'truncated_input': False} 

3. Input = 'ok'
{'priority': 'P3', 'confidence': 0.8552, 'all_probs': {'P1': 0.021132875233888626, 'P2': 0.12366613000631332, 'P3': 0.8552010655403137}, 'routing_decision': 'auto_route', 'truncated_input': False} 

4. Input length = 3000 (truncated to 2000)
truncated_input = True
priority = P2



### Task 4 (20 pts) — Batch-run on the fresh Day211 test set + NRA

Run the wrapper on all 20 tickets in `day211_test_set`. Build a results DataFrame with columns: `text, true_priority, predicted_priority, confidence, routing_decision, correct`.

Then compute, from the DataFrame (not from memory):
- Overall accuracy on this fresh set
- `auto_route` accuracy vs `human_review` accuracy — **does the confidence threshold actually catch more of the wrong predictions?** This is the real test of whether Day210's threshold is doing its job on brand-new data.
- Human-review rate (% of tickets flagged for a human)

Write ONE NRA block (Number + Reason + Action) about whether this wrapper is currently safe to hand tickets to unsupervised, or whether the human-review rate needs to go up/down.


In [18]:
# ── TASK 4 (20 pts): Batch‑run on the fresh Day211 test set + NRA ──
# Goal: Evaluate the wrapper on all 20 tickets, compute accuracy splits by
#       routing decision, and write an NRA about the wrapper's safety.
# Method: Loop over day211_test_set, collect predictions into a DataFrame,
#         compute overall accuracy, auto_route accuracy, human_review accuracy,
#         and the human_review rate.

import pandas as pd

# Build results DataFrame
results_list = []
for ticket in day211_test_set:
    text = ticket["text"]
    true_label = ticket["true_priority"]
    pred = safe_predict_ticket_priority(text)
    # If error occurred (should not happen for valid strings), skip or handle
    if "error" in pred:
        continue
    results_list.append({
        "text": text,
        "true_priority": true_label,
        "predicted_priority": pred["priority"],
        "confidence": pred["confidence"],
        "routing_decision": pred["routing_decision"],
        "correct": (true_label == pred["priority"])
    })

results_df = pd.DataFrame(results_list)

# Compute metrics
overall_acc = results_df["correct"].mean()
auto_mask = results_df["routing_decision"] == "auto_route"
human_mask = results_df["routing_decision"] == "human_review"

auto_acc = results_df[auto_mask]["correct"].mean() if auto_mask.any() else None
human_acc = results_df[human_mask]["correct"].mean() if human_mask.any() else None
human_rate = human_mask.mean()

print("--- Task 4: Batch evaluation results ---")
print(f"Overall accuracy: {overall_acc:.2%}")
print(f"Auto-route accuracy: {auto_acc:.2%} (on {auto_mask.sum()} tickets)")
print(f"Human-review accuracy: {human_acc:.2%} (on {human_mask.sum()} tickets)")
print(f"Human-review rate: {human_rate:.2%}")

# Show a few examples
print("\nSample predictions:")
print(results_df[["true_priority", "predicted_priority", "confidence", "routing_decision", "correct"]].head())

--- Task 4: Batch evaluation results ---
Overall accuracy: 70.00%
Auto-route accuracy: 68.75% (on 16 tickets)
Human-review accuracy: 75.00% (on 4 tickets)
Human-review rate: 20.00%

Sample predictions:
  true_priority predicted_priority  confidence routing_decision  correct
0            P3                 P3      0.9854       auto_route     True
1            P1                 P1      0.8135       auto_route     True
2            P2                 P3      0.6847       auto_route    False
3            P3                 P3      0.9164       auto_route     True
4            P2                 P2      0.5603     human_review     True


**Task 4 NRA:**

- **Number:** Overall accuracy = 70.00%; auto‑route accuracy = 68.75% (16 tickets), human‑review accuracy = 75.00% (4 tickets).
- **Reason:** On this fresh 20‑ticket batch, the auto‑routed (high‑confidence) group performed *worse* (68.75%) than the human‑review (low‑confidence) group (75.00%) — the opposite of the pattern seen on Day210’s eval where lower‑confidence predictions were more often wrong. However, with only 4 human‑review tickets, a single flip would change that group’s accuracy by ±25 points; the reversal could be noise rather than a real breakdown of the threshold.
- **Action:** Do **not** yet trust the 0.60 threshold as a safety gate. Before deploying, run a larger fresh batch (>50 tickets) to determine whether the reversal persists. If auto‑route accuracy remains below human‑review accuracy, raise the threshold to 0.70 or abandon automatic routing entirely. For the capstone, treat every prediction as human‑reviewable until this is resolved.

### Task 5 (15 pts) — Package as an importable module

Write the FINAL, cleaned-up version of everything above (config constants, model loading, `predict_ticket_priority`) into a single file `day211_inference_wrapper.py` using `%%writefile`. It must:
- Be importable with no side effects at import time other than loading the model (no test prints running automatically)
- Expose exactly one public function: `predict_ticket_priority(text)`
- Include a module-level docstring and a function docstring

Then, in a NEW cell, actually `import` it (`from day211_inference_wrapper import predict_ticket_priority`) and call it once, to prove the packaged version works standalone — this is what Days212-216 will do.


In [19]:
%%writefile day211_inference_wrapper.py
"""
day211_inference_wrapper.py

Production inference wrapper for the TeleServe India ticket priority classifier.
Loads the 4‑bit DistilBERT + LoRA adapter and provides a single public function
`predict_ticket_priority(text)` that returns a structured decision.
"""

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification, BitsAndBytesConfig
from peft import PeftModel
import os

# ---- Constants (pulled from Day210 evaluation) ----
BASE_MODEL = "distilbert-base-uncased"
ADAPTER_PATH = "/content/drive/MyDrive/teleserve_lora_adapter_209"   # adjust if Drive mounted differently
ID2LABEL = {0: "P1", 1: "P2", 2: "P3"}
LABEL2ID = {v: k for k, v in ID2LABEL.items()}
CONFIDENCE_THRESHOLD = 0.60
MAX_INPUT_LENGTH = 2000

# ---- Lazy loading (load on first call) ----
_model = None
_tokenizer = None

def _load_model():
    """Load the model and tokenizer once."""
    global _model, _tokenizer
    if _model is not None:
        return
    # Tokenizer
    _tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    # Quantization config
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        llm_int8_skip_modules=["pre_classifier", "classifier"],
    )
    # Base model
    base_model = AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL,
        num_labels=3,
        id2label=ID2LABEL,          # correct: int → str
        label2id=LABEL2ID,          # correct: str → int
        quantization_config=bnb_config,
        device_map="auto",
    )
    # Adapter
    _model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
    _model.eval()
    torch.cuda.empty_cache()


def predict_ticket_priority(text: str) -> dict:
    """
    Predict priority and confidence for a ticket text.

    Args:
        text (str): Raw customer ticket text.

    Returns:
        dict: {
            "priority": str,           # "P1", "P2", or "P3"
            "confidence": float,       # softmax probability of predicted class
            "all_probs": dict,         # {"P1": prob, "P2": prob, "P3": prob}
            "routing_decision": str,   # "auto_route" or "human_review"
            "truncated_input": bool,   # True if input was longer than 2000 chars
            "error": str or None       # None if successful, else error message
        }
    """
    # Load model if not already loaded
    _load_model()

    # ---- Input validation ----
    if not isinstance(text, str):
        return {
            "error": "input must be a string",
            "priority": None,
            "confidence": None,
            "all_probs": None,
            "routing_decision": None,
            "truncated_input": False
        }
    stripped = text.strip()
    if len(stripped) == 0:
        return {
            "error": "input too short (empty or whitespace)",
            "priority": None,
            "confidence": None,
            "all_probs": None,
            "routing_decision": None,
            "truncated_input": False
        }
    truncated = False
    if len(stripped) > MAX_INPUT_LENGTH:
        stripped = stripped[:MAX_INPUT_LENGTH]
        truncated = True

    # ---- Inference ----
    inputs = _tokenizer(stripped, truncation=True, padding=True, max_length=64, return_tensors="pt")
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}

    with torch.no_grad():
        logits = _model(**inputs).logits
        probs = F.softmax(logits, dim=-1).cpu().numpy()[0]

    # Map probabilities to label names
    probs_dict = {ID2LABEL[i]: float(probs[i]) for i in range(3)}
    pred_label = max(probs_dict, key=probs_dict.get)
    confidence = probs_dict[pred_label]
    routing = "human_review" if confidence < CONFIDENCE_THRESHOLD else "auto_route"

    return {
        "priority": pred_label,
        "confidence": round(confidence, 4),
        "all_probs": probs_dict,
        "routing_decision": routing,
        "truncated_input": truncated,
        "error": None
    }

Overwriting day211_inference_wrapper.py


In [20]:
# ── TASK 5 (continued): Prove the packaged module works ────────────
# Goal: Import the module and call the function to confirm it loads and runs.
# Method: Import from the file just written and test on a sample ticket.

from day211_inference_wrapper import predict_ticket_priority

test_text = "Network down in my area since morning, urgent!"
result = predict_ticket_priority(test_text)
print("Module import successful. Test result:")
print(result)

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cpu/ops.py:80: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cpu/ops.py:132: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Module import successful. Test result:
{'priority': 'P3', 'confidence': 0.5529, 'all_probs': {'P1': 0.38781964778900146, 'P2': 0.05924523249268532, 'P3': 0.5529350638389587}, 'routing_decision': 'human_review', 'truncated_input': False, 'error': None}


### Bonus (10★) — Latency check

Time 10 calls to `predict_ticket_priority()` (use `time.perf_counter()`, not `%timeit` — you want per-call numbers, not aggregated). Compute avg latency per call in ms.

Write ONE NRA on whether this latency is acceptable for a chatbot the capstone will build (a reasonable real-time chat budget is roughly <2000ms per turn, since the wrapper is only one part of the agent's total response time) — ground the Number in your actual printed avg, not this general guidance.


In [21]:
# ── BONUS (10★): Latency check ──────────────────────────────────────
# Goal: Measure per‑call inference latency to ensure it meets the <2000 ms budget.
# Method: Run 10 calls with time.perf_counter(), compute average in ms.

import time

num_calls = 10
test_texts = [ticket["text"] for ticket in day211_test_set[:num_calls]]  # use first 10 tickets

latencies = []
for text in test_texts:
    start = time.perf_counter()
    _ = predict_ticket_priority(text)
    end = time.perf_counter()
    latencies.append((end - start) * 1000)   # ms

avg_latency = sum(latencies) / len(latencies)

print(f"Average inference latency over {num_calls} calls: {avg_latency:.2f} ms")
print(f"Min: {min(latencies):.2f} ms, Max: {max(latencies):.2f} ms")

Average inference latency over 10 calls: 1489.91 ms
Min: 952.62 ms, Max: 3262.66 ms


**Bonus NRA:**

- **Number:** Average latency = 1489.91 ms, max latency = 3262.66 ms (one call exceeded the 2000 ms budget by 63%).
- **Reason:** The measured latency is **CPU‑typical** for a 4‑bit DistilBERT forward pass – the earlier `Model device: cpu` output confirms the model is not running on the T4 GPU. On GPU, latency would be ~10‑20× lower, so the current average (~1.5s) is not the model’s true performance ceiling. The tail latency (one call >3.2s) would be unacceptable in a real‑time chatbot if it happened repeatedly.
- **Action:** Before the capstone, **force the runtime to use a T4 GPU** (Runtime → Change runtime type → GPU) and re‑measure latency. Only after a fresh GPU‑timed average <200ms can we call “no optimisation needed.” If even on GPU tail latency spikes, implement a timeout fallback that routes to a human after 1.5s.

## Interview Framing

**"Walk me through how you took a fine-tuned model and made it production-ready."**

> "After fine-tuning and honestly evaluating a DistilBERT ticket-classifier with LoRA — including finding it only generalizes 60% of the time on unseen phrasing versus 100% on validation data — I didn't just ship the raw model. I built an inference wrapper: one function with a strict contract, input validation that never crashes on bad data, and a confidence threshold derived from real evaluation numbers, not a guess, that automatically routes low-confidence predictions to a human instead of silently guessing wrong. I packaged it as an importable module so it's a clean dependency for the downstream agent, not a copy-pasted notebook cell. That's the difference between a model that scores well in a notebook and a component another engineer — or another AI agent — can actually call in production."

This answer works because it shows the **generalization-honesty finding carrying forward into a design decision** (the threshold), not just being reported and forgotten.


## Section 4 — Scoring Rubric

| Task | Points | Criteria |
|---|---|---|
| Task 1 — Load model + adapter, config | 20 | Correct `BitsAndBytesConfig` incl. `llm_int8_skip_modules`; tokenizer from base model not adapter path; `CONFIDENCE_THRESHOLD` set from Day210's real printed number (0.60), not guessed |
| Task 2 — Core predict function | 25 | Correct return shape (priority/confidence/all_probs/routing_decision); uses `torch.no_grad()`; softmax applied correctly; tested on real tickets |
| Task 3 — Input validation | 20 | All 4 edge cases handled without raising; truncation flag set correctly; error dicts structured consistently |
| Task 4 — Batch run + NRA | 20 | Results DataFrame built correctly from real predictions; accuracy split by routing_decision computed from the DataFrame; NRA Number matches printed output exactly, Reason states mechanism only, Action is specific |
| Task 5 — Packaged module | 15 | `%%writefile` produces a self-contained, importable file; no side-effect prints at import; standalone import + call proven in a fresh cell |
| **Subtotal** | **100** | |
| Bonus — Latency | +10★ | Real per-call timing (not `%timeit` aggregate); NRA Number matches printed avg; Reason distinguishes compute cost vs overhead; Action gives a concrete recommendation for Day212+ |

**Grading note (standing discipline, Day210 lesson):** environment/infra correctness (does the model load, does the module import cleanly) and NRA-writing correctness (Number from real output, Reason stops at mechanism, Action is specific) are graded and fixed **separately** — a working notebook with sloppy NRA prose is not a passing Task 4, and clean NRA prose describing numbers that don't match printed output is not either.
